# Aula 09 — Técnicas Estatísticas de Predição
**Ciência de Dados · Univassouras · Prof. Mesc. Diego Ramos Inácio**

---

Este notebook acompanha a apresentação da Aula 09. Você vai encontrar:

| # | Conteúdo |
|---|----------|
| 1 | Configuração do ambiente |
| 2 | **Exemplo 1** — Regressão Linear Simples (salário × experiência) |
| 3 | **Exemplo 2** — Regressão Linear Múltipla (preço de imóvel) |
| 4 | **Exemplo 3** — Regressão Logística (aprovação de crédito) |
| 5 | **Exemplo 4** — Árvore de Decisão vs. Random Forest |
| 6 | Comparativo de métricas entre modelos |
| 7 | **Exercício** — sua vez! |

> **Dica:** execute cada célula com `Shift + Enter`.

## 0 · Configuração do Ambiente

Instale as dependências caso ainda não tenha (remova o `#` e execute).

In [ ]:
# !pip install numpy pandas matplotlib scikit-learn --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Ambiente pronto!')

---
## 1 · Exemplo 1 — Regressão Linear Simples
### Problema: prever salário a partir dos anos de experiência

**Variável alvo (Y):** Salário (R$ mil)  
**Preditor (X):** Anos de experiência  
**Modelo:** Ŷ = β₀ + β₁ · X

> **Intuição:** quanto mais anos de experiência, maior o salário esperado. O modelo encontra a reta que melhor representa essa relação.

In [ ]:
# ── Dados ────────────────────────────────────────────────────
experiencia = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15,
               1.5, 3.5, 7, 9, 11, 13, 4.5, 6.5, 2.5, 5.5]
salario     = [3.8, 4.5, 5.2, 5.8, 6.9, 7.4, 9.2, 10.8,
               12.5, 15.0, 4.1, 5.5, 8.3, 10.0, 11.8, 13.5,
               6.0, 7.9, 4.8, 7.2]

df1 = pd.DataFrame({'experiencia': experiencia, 'salario': salario})
print('Estatísticas descritivas:')
df1.describe().round(2)

In [ ]:
# ── Divisão treino / teste (80% / 20%) ───────────────────────
X1 = df1[['experiencia']]
y1 = df1['salario']

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.2, random_state=42
)
print(f'Treino: {len(X1_train)} amostras | Teste: {len(X1_test)} amostras')

In [ ]:
# ── Treinar e avaliar ─────────────────────────────────────────
modelo1 = LinearRegression()
modelo1.fit(X1_train, y1_train)

y1_pred = modelo1.predict(X1_test)

b0 = modelo1.intercept_
b1 = modelo1.coef_[0]
mae  = mean_absolute_error(y1_test, y1_pred)
rmse = np.sqrt(mean_squared_error(y1_test, y1_pred))
r2   = r2_score(y1_test, y1_pred)

print(f'Equação: Salário = {b0:.2f} + {b1:.2f} × Experiência')
print(f'\nInterpretação:')
print(f'  β₀ = {b0:.2f} → salário base (0 anos de exp.): R$ {b0:.0f} mil')
print(f'  β₁ = {b1:.2f} → cada ano extra vale R$ {b1*1000:.0f} a mais')
print(f'\nMétricas no conjunto de teste:')
print(f'  MAE  = {mae:.3f} (erro médio de R$ {mae:.2f}k)')
print(f'  RMSE = {rmse:.3f}')
print(f'  R²   = {r2:.4f} ({r2*100:.1f}% da variação explicada)')

In [ ]:
# ── Visualização ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

X_linha = np.linspace(0, 16, 100).reshape(-1, 1)
y_linha = modelo1.predict(X_linha)

# Gráfico 1: reta ajustada
axes[0].scatter(X1_train, y1_train, color='#0d9488', label='Treino', s=60, alpha=.8)
axes[0].scatter(X1_test,  y1_test,  color='#e11d48', label='Teste',  s=80, marker='D')
axes[0].plot(X_linha, y_linha, 'k--', lw=2, label='Reta ajustada')
for xi, yi, yp in zip(X1_test.values.flatten(), y1_test, y1_pred):
    axes[0].plot([xi, xi], [yi, yp], color='gray', lw=1, ls=':')
axes[0].set_xlabel('Experiência (anos)')
axes[0].set_ylabel('Salário (R$ mil)')
axes[0].set_title(f'Regressão Linear Simples\nR² = {r2:.3f}')
axes[0].legend()

# Gráfico 2: real vs previsto
axes[1].scatter(y1_test, y1_pred, color='#2563eb', s=80, edgecolors='white', lw=1)
diag = np.linspace(y1_test.min(), y1_test.max(), 50)
axes[1].plot(diag, diag, 'r--', lw=1.5, label='Predição perfeita')
axes[1].set_xlabel('Valor Real (R$ mil)')
axes[1].set_ylabel('Valor Previsto (R$ mil)')
axes[1].set_title('Real vs Previsto\n(ideal: pontos na diagonal)')
axes[1].legend()

plt.suptitle('Exemplo 1 — Regressão Linear Simples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 2 · Exemplo 2 — Regressão Linear Múltipla
### Problema: prever o preço de um imóvel

**Variável alvo (Y):** Preço (R$ mil)  
**Preditores (X):** Área (m²), Número de quartos, Distância ao centro (km)  
**Modelo:** Ŷ = β₀ + β₁·Área + β₂·Quartos + β₃·Distância

> **Intuição:** o preço depende de vários fatores ao mesmo tempo. A regressão múltipla captura o efeito **parcial** de cada variável, mantendo as outras constantes.

In [ ]:
# ── Dados sintéticos de imóveis ───────────────────────────────
n = 120
area       = np.random.uniform(40, 200, n)
quartos    = np.random.choice([1, 2, 3, 4], n, p=[.15, .35, .35, .15])
distancia  = np.random.uniform(1, 25, n)
ruido      = np.random.normal(0, 30, n)

# Fórmula verdadeira: preço = 80 + 2.5·área + 40·quartos - 8·distância + ruído
preco = 80 + 2.5 * area + 40 * quartos - 8 * distancia + ruido

df2 = pd.DataFrame({
    'area': area, 'quartos': quartos,
    'distancia_centro_km': distancia, 'preco_mil': preco
})

print('Amostra dos dados:')
df2.head(8).round(1)

In [ ]:
# ── Correlações ───────────────────────────────────────────────
print('Correlação com o preço:')
print(df2.corr()[['preco_mil']].round(3))

# Visualização rápida
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, cor in zip(axes,
                         ['area', 'quartos', 'distancia_centro_km'],
                         ['#0d9488', '#2563eb', '#e11d48']):
    ax.scatter(df2[col], df2['preco_mil'], color=cor, alpha=.5, s=25)
    ax.set_xlabel(col)
    ax.set_ylabel('Preço (R$ mil)')
    r = df2[col].corr(df2['preco_mil'])
    ax.set_title(f'Correlação = {r:.2f}')
plt.suptitle('Relação de cada preditor com o preço', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Treinar e avaliar ─────────────────────────────────────────
X2 = df2[['area', 'quartos', 'distancia_centro_km']]
y2 = df2['preco_mil']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42
)

modelo2 = LinearRegression()
modelo2.fit(X2_train, y2_train)
y2_pred = modelo2.predict(X2_test)

print('Coeficientes encontrados:')
coef_df = pd.DataFrame({
    'Variável': ['intercepto'] + list(X2.columns),
    'Coeficiente (β)': [modelo2.intercept_] + list(modelo2.coef_),
    'Valor real': [80, 2.5, 40, -8]
})
print(coef_df.to_string(index=False))

print(f'\nMétricas:')
print(f'  MAE  = {mean_absolute_error(y2_test, y2_pred):.2f} mil')
print(f'  RMSE = {np.sqrt(mean_squared_error(y2_test, y2_pred)):.2f} mil')
print(f'  R²   = {r2_score(y2_test, y2_pred):.4f}')

print('\nInterpretação (mantendo as demais constantes):')
print(f'  +1 m² de área → R$ {modelo2.coef_[0]*1000:.0f} a mais no preço')
print(f'  +1 quarto     → R$ {modelo2.coef_[1]*1000:.0f} a mais no preço')
print(f'  +1 km do centro → R$ {modelo2.coef_[2]*1000:.0f} a menos no preço')

---
## 3 · Exemplo 3 — Regressão Logística
### Problema: prever aprovação de crédito (sim/não)

**Variável alvo (Y):** Aprovado (1) ou Negado (0)  
**Preditores (X):** Renda mensal, Score de crédito, Dívida atual  
**Saída:** Probabilidade de aprovação entre 0 e 1

> **Intuição:** em vez de prever um número, queremos saber a **chance** de um cliente ser aprovado. A função sigmoide transforma qualquer valor em uma probabilidade.

In [ ]:
# ── Dados de crédito ─────────────────────────────────────────
n = 200
renda = np.random.uniform(1.5, 20, n)       # R$ mil
score = np.random.uniform(300, 900, n)       # score 300–900
divida = np.random.uniform(0, 15, n)          # R$ mil

# Probabilidade real baseada nas variáveis
logit = -4 + 0.15 * renda + 0.006 * score - 0.12 * divida
prob  = 1 / (1 + np.exp(-logit))
aprovado = (np.random.uniform(0, 1, n) < prob).astype(int)

df3 = pd.DataFrame({
    'renda_mil': renda, 'score_credito': score,
    'divida_mil': divida, 'aprovado': aprovado
})

print('Distribuição das classes:')
print(df3['aprovado'].value_counts().rename({0: 'Negado', 1: 'Aprovado'}))

df3.head(6).round(2)

In [ ]:
# ── Treinar regressão logística ───────────────────────────────
X3 = df3[['renda_mil', 'score_credito', 'divida_mil']]
y3 = df3['aprovado']

X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.25, random_state=42
)

# Padronizar (importante para regressão logística)
scaler = StandardScaler()
X3_train_s = scaler.fit_transform(X3_train)
X3_test_s  = scaler.transform(X3_test)

modelo3 = LogisticRegression(random_state=42)
modelo3.fit(X3_train_s, y3_train)

y3_pred  = modelo3.predict(X3_test_s)
y3_proba = modelo3.predict_proba(X3_test_s)[:, 1]

print('Relatório de classificação:')
print(classification_report(y3_test, y3_pred,
                             target_names=['Negado', 'Aprovado']))
print(f'Acurácia: {accuracy_score(y3_test, y3_pred):.1%}')

In [ ]:
# ── Visualização: probabilidades e matriz de confusão ─────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribuição das probabilidades por classe real
for cls, cor, nome in [(0, '#e11d48', 'Negado'), (1, '#0d9488', 'Aprovado')]:
    mask = y3_test == cls
    axes[0].hist(y3_proba[mask], bins=20, alpha=.6, color=cor, label=nome)
axes[0].axvline(0.5, color='black', ls='--', lw=1.5, label='Limiar = 0,5')
axes[0].set_xlabel('Probabilidade de aprovação')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição das probabilidades\npor classe real')
axes[0].legend()

# Matriz de confusão
cm = confusion_matrix(y3_test, y3_pred)
im = axes[1].imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, cm[i, j], ha='center', va='center',
                     fontsize=18, fontweight='bold',
                     color='white' if cm[i, j] > cm.max()/2 else 'black')
axes[1].set_xticks([0, 1]); axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(['Negado', 'Aprovado'])
axes[1].set_yticklabels(['Negado', 'Aprovado'])
axes[1].set_xlabel('Previsto')
axes[1].set_ylabel('Real')
axes[1].set_title('Matriz de Confusão')

plt.suptitle('Exemplo 3 — Regressão Logística', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Simulador: probabilidade para um novo cliente ─────────────
novos_clientes = pd.DataFrame({
    'renda_mil':     [3.0,  8.5, 15.0,  2.0],
    'score_credito': [450, 720,  850,   380],
    'divida_mil':    [5.0,  2.0,  0.5,  12.0]
})

novos_s = scaler.transform(novos_clientes)
probas  = modelo3.predict_proba(novos_s)[:, 1]

novos_clientes['prob_aprovacao'] = probas.round(3)
novos_clientes['decisao'] = ['✅ Aprovado' if p >= 0.5 else '❌ Negado' for p in probas]
print('Simulação — novos clientes:')
novos_clientes

---
## 4 · Exemplo 4 — Árvore de Decisão vs. Random Forest
### Problema: prever consumo de energia elétrica (kWh/mês)

**Variável alvo (Y):** Consumo (kWh/mês)  
**Preditores (X):** Área do imóvel, N° de moradores, N° de aparelhos, Tem ar-condicionado

> **Intuição:** a árvore aprende regras do tipo "se a área > 80m² e tem ar-cond, então consumo ≈ X". O Random Forest combina centenas dessas árvores para reduzir os erros.

In [ ]:
# ── Dados de consumo elétrico ─────────────────────────────────
n = 250
area_imovel  = np.random.uniform(30, 180, n)
moradores    = np.random.randint(1, 6, n)
aparelhos    = np.random.randint(3, 20, n)
tem_arcond   = np.random.choice([0, 1], n, p=[.4, .6])
ruido4       = np.random.normal(0, 20, n)

# Fórmula: consumo = 50 + 1.2·área + 15·moradores + 5·aparelhos + 80·ar-cond
consumo = 50 + 1.2*area_imovel + 15*moradores + 5*aparelhos + 80*tem_arcond + ruido4

df4 = pd.DataFrame({
    'area_m2': area_imovel, 'moradores': moradores,
    'aparelhos': aparelhos, 'ar_condicionado': tem_arcond,
    'consumo_kwh': consumo
})

X4 = df4.drop(columns='consumo_kwh')
y4 = df4['consumo_kwh']

X4_train, X4_test, y4_train, y4_test = train_test_split(
    X4, y4, test_size=0.2, random_state=42
)

print(f'Consumo médio: {consumo.mean():.1f} kWh/mês')
print(f'Variação:      {consumo.min():.0f} – {consumo.max():.0f} kWh/mês')

In [ ]:
# ── Treinar Árvore de Decisão e Random Forest ─────────────────
arvore = DecisionTreeRegressor(max_depth=4, random_state=42)
arvore.fit(X4_train, y4_train)

floresta = RandomForestRegressor(n_estimators=100, max_depth=6, random_state=42)
floresta.fit(X4_train, y4_train)

y4_pred_arv = arvore.predict(X4_test)
y4_pred_flo = floresta.predict(X4_test)

# Comparativo
resultados = pd.DataFrame({
    'Modelo': ['Árvore de Decisão', 'Random Forest'],
    'MAE':  [mean_absolute_error(y4_test, y4_pred_arv),
              mean_absolute_error(y4_test, y4_pred_flo)],
    'RMSE': [np.sqrt(mean_squared_error(y4_test, y4_pred_arv)),
              np.sqrt(mean_squared_error(y4_test, y4_pred_flo))],
    'R²':   [r2_score(y4_test, y4_pred_arv),
              r2_score(y4_test, y4_pred_flo)]
})
print(resultados.round(3).to_string(index=False))

In [ ]:
# ── Visualização: comparativo e importância dos atributos ──────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

diag = np.linspace(y4_test.min(), y4_test.max(), 50)

# Real vs Previsto — Árvore
r2_arv = r2_score(y4_test, y4_pred_arv)
axes[0].scatter(y4_test, y4_pred_arv, color='#d97706', alpha=.6, s=30)
axes[0].plot(diag, diag, 'r--', lw=1.5)
axes[0].set_title(f'Árvore de Decisão\nR² = {r2_arv:.3f}')
axes[0].set_xlabel('Real (kWh)')
axes[0].set_ylabel('Previsto (kWh)')

# Real vs Previsto — Random Forest
r2_flo = r2_score(y4_test, y4_pred_flo)
axes[1].scatter(y4_test, y4_pred_flo, color='#15803d', alpha=.6, s=30)
axes[1].plot(diag, diag, 'r--', lw=1.5)
axes[1].set_title(f'Random Forest\nR² = {r2_flo:.3f}')
axes[1].set_xlabel('Real (kWh)')
axes[1].set_ylabel('Previsto (kWh)')

# Importância dos atributos (Random Forest)
importancias = pd.Series(floresta.feature_importances_, index=X4.columns)
importancias.sort_values().plot(kind='barh', ax=axes[2], color='#2563eb')
axes[2].set_title('Importância dos Atributos\n(Random Forest)')
axes[2].set_xlabel('Importância relativa')

plt.suptitle('Exemplo 4 — Árvore de Decisão vs Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Regras aprendidas pela Árvore (profundidade 3) ────────────
arvore_vis = DecisionTreeRegressor(max_depth=3, random_state=42)
arvore_vis.fit(X4_train, y4_train)
print('Regras da Árvore de Decisão (profundidade 3):')
print(export_text(arvore_vis, feature_names=list(X4.columns)))

---
## 5 · Comparativo de Métricas entre Modelos

Vamos reunir os resultados dos três exemplos de **regressão** em um painel comparativo.

In [ ]:
# ── Regressão Linear no problema de energia (base de comparação)
reg_lin_4 = LinearRegression()
reg_lin_4.fit(X4_train, y4_train)
y4_pred_lin = reg_lin_4.predict(X4_test)

comparativo = pd.DataFrame({
    'Problema': ['Salário (linear simples)',
                 'Preço Imóvel (linear múltipla)',
                 'Consumo — Regressão Linear',
                 'Consumo — Árvore de Decisão',
                 'Consumo — Random Forest'],
    'MAE': [
        mean_absolute_error(y1_test, y1_pred),
        mean_absolute_error(y2_test, y2_pred),
        mean_absolute_error(y4_test, y4_pred_lin),
        mean_absolute_error(y4_test, y4_pred_arv),
        mean_absolute_error(y4_test, y4_pred_flo)
    ],
    'RMSE': [
        np.sqrt(mean_squared_error(y1_test, y1_pred)),
        np.sqrt(mean_squared_error(y2_test, y2_pred)),
        np.sqrt(mean_squared_error(y4_test, y4_pred_lin)),
        np.sqrt(mean_squared_error(y4_test, y4_pred_arv)),
        np.sqrt(mean_squared_error(y4_test, y4_pred_flo))
    ],
    'R²': [
        r2_score(y1_test, y1_pred),
        r2_score(y2_test, y2_pred),
        r2_score(y4_test, y4_pred_lin),
        r2_score(y4_test, y4_pred_arv),
        r2_score(y4_test, y4_pred_flo)
    ]
})

print('Painel comparativo:')
comparativo.round(3)

In [ ]:
# ── Gráfico comparativo de R² ─────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
cores = ['#0d9488', '#2563eb', '#d97706', '#d97706', '#15803d']
bars = ax.barh(comparativo['Problema'], comparativo['R²'],
               color=cores, height=0.6, edgecolor='white')
ax.set_xlabel('R² (quanto maior, melhor)')
ax.set_xlim(0, 1.05)
ax.axvline(0.7, color='gray', ls='--', lw=1, label='R² = 0,70 (referência)')
ax.axvline(0.9, color='green', ls='--', lw=1, label='R² = 0,90 (excelente)')
for bar, val in zip(bars, comparativo['R²']):
    ax.text(val + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.set_title('Comparativo de R² entre modelos e problemas', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6 · EXERCÍCIO — Sua Vez!

### Contexto
Uma rede de postos de combustível quer prever o **volume de vendas diário (litros)** de gasolina em cada posto. O gestor acredita que as principais variáveis explicativas são:

| Variável | Descrição |
|----------|-----------|
| `fluxo_veiculos` | Número de veículos que passam por dia na via |
| `preco_litro` | Preço médio do litro (R$) |
| `concorrentes_raio` | Número de postos concorrentes em raio de 2 km |
| `fim_de_semana` | 1 se é fim de semana, 0 se é dia útil |

### Tarefas

1. **Explore os dados** — veja as estatísticas descritivas e as correlações
2. **Treine um modelo de Regressão Linear Múltipla**
3. **Calcule as métricas** (MAE, RMSE, R²)
4. **Interprete os coeficientes** — qual variável tem maior impacto?
5. **Bonus:** compare com um Random Forest — qual é melhor?

In [ ]:
# ── Dados do exercício (não altere esta célula) ────────────────
np.random.seed(7)
n_ex = 300

fluxo = np.random.randint(500, 8000, n_ex)
preco = np.random.uniform(5.2, 6.8, n_ex)
concorr = np.random.randint(0, 6, n_ex)
fds = np.random.choice([0, 1], n_ex, p=[.71, .29])
ruido_ex = np.random.normal(0, 80, n_ex)

vendas = (800 + 0.08 * fluxo - 150 * preco
          - 40 * concorr + 120 * fds + ruido_ex).clip(0)

df_ex = pd.DataFrame({
    'fluxo_veiculos':     fluxo,
    'preco_litro':        preco.round(2),
    'concorrentes_raio':  concorr,
    'fim_de_semana':      fds,
    'vendas_litros':      vendas.round(0)
})

print('Dados do exercício:')
df_ex.head(10)

In [ ]:
# ── TAREFA 1: Explore os dados ────────────────────────────────
# Dica: use .describe() e .corr()

# Seu código aqui:


In [ ]:
# ── TAREFA 2 e 3: Regressão Linear Múltipla + Métricas ────────
# Dica: separe X e y, faça train_test_split, treine LinearRegression

# Definindo X e y:
X_ex = df_ex.drop(columns='vendas_litros')
y_ex = df_ex['vendas_litros']

# Complete o código abaixo:
X_ex_train, X_ex_test, y_ex_train, y_ex_test = train_test_split(
    X_ex, y_ex, test_size=_____, random_state=42   # <-- defina o tamanho do teste
)

modelo_ex = _____()                                # <-- qual modelo usar?
modelo_ex.fit(_____, _____)                        # <-- treine com treino

y_ex_pred = modelo_ex.predict(_____)              # <-- preveja no teste

# Métricas:
mae_ex  = mean_absolute_error(_____, _____)
rmse_ex = np.sqrt(mean_squared_error(_____, _____))
r2_ex   = r2_score(_____, _____)

print(f'MAE  = {mae_ex:.2f} litros')
print(f'RMSE = {rmse_ex:.2f} litros')
print(f'R²   = {r2_ex:.4f}')

In [ ]:
# ── TAREFA 4: Interprete os coeficientes ─────────────────────
# Crie um DataFrame com as variáveis e seus coeficientes
# Qual variável tem maior impacto positivo? E negativo?

# Seu código aqui:


In [ ]:
# ── TAREFA 5 (Bonus): Comparação com Random Forest ────────────
# Treine um RandomForestRegressor e compare o R² com a regressão linear

# Seu código aqui:


In [ ]:
# ── Visualize os resultados do exercício ──────────────────────
# Crie pelo menos um gráfico para apresentar suas descobertas

# Seu código aqui:


---
### Perguntas para reflexão

Responda nas células abaixo (texto livre — clique duas vezes para editar):

**1.** Qual variável tem maior impacto nas vendas? Faz sentido com a realidade?

> *Sua resposta aqui...*

**2.** O modelo de Regressão Linear foi suficiente, ou o Random Forest melhorou muito o R²? Por que isso pode acontecer?

> *Sua resposta aqui...*

**3.** Se você fosse o gestor da rede de postos, que ação tomaria com base nos coeficientes do modelo?

> *Sua resposta aqui...*

---
**Ciência de Dados · Aula 09 · Prof. Mesc. Diego Ramos Inácio · Univassouras · 2026**